# 05 — Controlled Sensor Corruption Robustness & SRM Ablation Study
**Course:** BCSE209P — Machine Learning Lab  
**Students:** P Naga Venkata Kushal (23BDS0133) & Koteru Tarun (23BDS0103)  
**Faculty Guide:** Dr. Dega Nagaraju  
**Repository:** [Reliability-Aware Anomaly Detection](https://github.com/nvk321/Reliability-Aware-Anomaly-Detection)

---
### Objectives
1. Demonstrate synthetic controlled sensor corruptions using `src.corruption.corruptor.SensorCorruptor`.
2. Inspect audited multi-seed robustness results across Gaussian noise, stuck sensors, and dynamic drift.
3. Evaluate the SRM ablation study (No SRM, -Temporal, -Graph, -Residual, Full SRM).
4. Discuss the physical actuator boundary (stuck sensors on zero-variance inactive pumps).


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))
from src.corruption.corruptor import SensorCorruptor
from src.visualization.plots import apply_scientific_style
apply_scientific_style()

print('Imports successful!')


### 1. Demonstrate Controlled Sensor Corruption
`SensorCorruptor` injects non-adversarial physical sensor degradations while preserving raw ground truth and creating a binary fault mask.


In [ ]:
corruptor = SensorCorruptor(seed=42)
dummy_data = np.sin(np.linspace(0, 20, 200))[:, None].repeat(5, axis=1)

# Apply 3 corruption modes to Channel 1
noisy_data, mask_n, _ = corruptor.apply_corruption(dummy_data, corruption_type='gaussian_noise', affected_channels=[1], severity=0.5)
drift_data, mask_d, _ = corruptor.apply_corruption(dummy_data, corruption_type='drift', affected_channels=[1], severity=1.0)
stuck_data, mask_s, _ = corruptor.apply_corruption(dummy_data, corruption_type='stuck_constant', affected_channels=[1])

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
axes[0].plot(dummy_data[:, 1], 'k--', label='Clean Signal', alpha=0.6)
axes[0].plot(noisy_data[:, 1], color='#DC2626', label='Gaussian Noise (sigma=0.5)')
axes[0].set_title('Corruption Mode 1: High Sensor Jitter (Noise)', fontweight='bold')
axes[0].legend(loc='upper right')

axes[1].plot(dummy_data[:, 1], 'k--', label='Clean Signal', alpha=0.6)
axes[1].plot(drift_data[:, 1], color='#D97706', label='Linear Calibration Drift (delta=1.0)')
axes[1].set_title('Corruption Mode 2: Calibration Drift', fontweight='bold')
axes[1].legend(loc='upper right')

axes[2].plot(dummy_data[:, 1], 'k--', label='Clean Signal', alpha=0.6)
axes[2].plot(stuck_data[:, 1], color='#7C3AED', label='Stuck Sensor (Frozen Reading)')
axes[2].set_title('Corruption Mode 3: Hardware Freeze (Stuck)', fontweight='bold')
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.show()


### 2. Load Audited Robustness Benchmark Summary
We inspect the multi-seed corruption summary comparing Baseline C and R-STNet.


In [ ]:
corr_summary_path = os.path.join('..', 'results', 'corruption', 'corruption_benchmark_summary.json')
with open(corr_summary_path, 'r') as f:
    corr_data = json.load(f)

print(f"Benchmark: {corr_data['benchmark']}")
print(f"Evaluated Regimes: {len(corr_data['regimes'])}")

rows = []
for r in corr_data['regimes']:
    rows.append({
        'Fault Mode': r['regime_name'],
        'Severity': r['severity'],
        'Baseline C F1': r['baseline_c_mean_f1'],
        'R-STNet F1': r['rstnet_mean_f1'],
        'F1 Delta': r['f1_improvement'],
        'R-STNet FPR': r['rstnet_mean_fpr'],
        'Reliability Sep': r['mean_reliability_separation'],
    })

df_corr = pd.DataFrame(rows)
df_corr


### 3. Load Audited Component Ablation Summary
We inspect the audited ablation experiments evaluating the individual components of the SRM.


In [ ]:
ablation_summary_path = os.path.join('..', 'results', 'ablations', 'ablation_summary.json')
with open(ablation_summary_path, 'r') as f:
    ablation_data = json.load(f)

ablation_rows = []
for a in ablation_data['ablations']:
    ablation_rows.append({
        'Ablation ID': a['id'],
        'Configuration': a['name'],
        'F1 Score': a['f1'],
        'Precision': a['precision'],
        'Recall': a['recall'],
        'FPR': a['fpr'],
        'F1 Gain vs Baseline C': a.get('f1_gain_vs_baseline_c', 0.0),
    })

df_ablation = pd.DataFrame(ablation_rows)
df_ablation


### Viva Voce Key Takeaways — Section G & H: Robustness, Ablation & Honest Results
1. **False Alarm Elimination:** Across 44,963 evaluation windows, R-STNet eliminates **14,898 false alarms** relative to Baseline C (a 37.8% relative reduction in FPR from 99.86% to 62.14%).
2. **Reliability Separation:** Under Gaussian noise and linear drift, the SRM achieves a clean-vs-corrupted separation of $\Delta r = +0.289$, correctly identifying 100% of synthetic sensor faults.
3. **Ablation Insight:** Removing Neighborhood Consistency (Ablation 4) drops F1 by $-0.038$, proving that inter-sensor physical redundancy is the strongest discriminator against false alarms.
4. **The Stuck-Sensor Boundary:** On inactive backup pumps (`P202`, `P204`), zero historical variance yields zero rate-of-change violation. Without supervisory PLC command telemetry, statistical SRM produces 0.0% fault recall on static actuators.
